# 03 — GraphRAG Experiment

End-to-end experiment with the self-built GraphRAG pipeline.

**Steps:**
1. Build: chunk → extract entities → build graph → communities → index
2. Explore the knowledge graph
3. Run sample queries (local, global, hybrid)
4. Compare with standard RAG results

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph_rag.pipeline import GraphRAGPipeline
from src.utils.logging_setup import setup_logging
setup_logging('INFO')

In [ ]:
# Build the full GraphRAG pipeline
# WARNING: This makes many LLM calls for entity extraction + summarisation.
# Consider running on a subset first.
pipe = GraphRAGPipeline()
pipe.build()

In [ ]:
# Explore the knowledge graph
kg = pipe.kg
print(f'Nodes: {kg.graph.number_of_nodes()}')
print(f'Edges: {kg.graph.number_of_edges()}')
print(f'\nTop 20 entities by degree:')
for name, degree in kg.top_entities(20):
    label = kg.graph.nodes[name].get('label', name)
    print(f'  {label}: {degree}')

In [ ]:
# Visualise a subgraph (requires pyvis)
from pyvis.network import Network
import networkx as nx

# Pick a central entity
ego = kg.get_neighbours('reinforcement learning', hops=2)
net = Network(notebook=True, height='600px', width='100%')
net.from_nx(ego)
net.show('graph_rl_ego.html')

In [ ]:
# Community summaries
print(f'Number of communities: {len(pipe.communities)}')
for c in pipe.communities[:5]:
    print(f'\nCommunity {c.community_id} ({len(c.nodes)} nodes):')
    print(f'  {c.summary[:200]}…')

In [ ]:
# Query — Local Search
from src.graph_rag.retriever import SearchMode

result = pipe.query('What is PPO?', mode=SearchMode.LOCAL)
print(f'Answer: {result.answer}')

In [ ]:
# Query — Global Search
result = pipe.query('What are the main themes in this research corpus?', mode=SearchMode.GLOBAL)
print(f'Answer: {result.answer}')